In [1]:
import random

class FoodDeliveryAgent:
    def __init__(self, start_location, restaurant_loc, customer_loc):
        # Sensors & Internal State
        self.current_location = start_location
        self.restaurant_loc = restaurant_loc
        self.customer_loc = customer_loc
        self.has_food = False
        
        # Performance Measure
        self.score = 100  # Starts with full rating score
        self.delivery_time = 0

    def sense_environment(self):
        """Sensor: Detects current traffic and weather conditions."""
        traffic = random.choice(["Light", "Moderate", "Heavy"])
        weather = random.choice(["Clear", "Rainy"])
        return traffic, weather

    def actuate_move(self, destination, traffic, weather):
        """Actuator: Moves the agent towards destination based on sensors."""
        delay = 1
        if traffic == "Heavy": delay += 2
        if weather == "Rainy": delay += 1
        
        self.delivery_time += delay
        self.current_location = destination
        
        # Penalty for delays in performance measure
        self.score -= (delay - 1) * 5

    def run_delivery(self):
        print(f"[Agent Started] At {self.current_location}")
        
        # Pickup Phase
        traffic, weather = self.sense_environment()
        print(f"Sensors -> Traffic: {traffic}, Weather: {weather}")
        self.actuate_move(self.restaurant_loc, traffic, weather)
        self.has_food = True
        print(f"[Pickup Complete] Food collected at {self.restaurant_loc}")

        # Delivery Phase
        traffic, weather = self.sense_environment()
        print(f"Sensors -> Traffic: {traffic}, Weather: {weather}")
        self.actuate_move(self.customer_loc, traffic, weather)
        print(f"[Delivery Complete] Food delivered to {self.customer_loc}")

        # Final Performance Evaluation
        print(f"--- Performance Report ---")
        print(f"Total Time Taken: {self.delivery_time} units")
        print(f"Final Customer Score: {self.score}/100")

# Run Agent Simulation
agent = FoodDeliveryAgent(start_location="Depot", restaurant_loc="Pizza Palace", customer_loc="123 Main St")
agent.run_delivery()

[Agent Started] At Depot
Sensors -> Traffic: Moderate, Weather: Clear
[Pickup Complete] Food collected at Pizza Palace
Sensors -> Traffic: Heavy, Weather: Clear
[Delivery Complete] Food delivered to 123 Main St
--- Performance Report ---
Total Time Taken: 4 units
Final Customer Score: 90/100


In [2]:
from collections import deque

def bfs_maze(grid, start, goal):
    """
    Finds the shortest path in a 2D grid using BFS.
    0 = Empty Path, 1 = Wall
    """
    rows, cols = len(grid), len(grid[0])
    queue = deque([(start[0], start[1], [start])])
    visited = {start}

    # Movement directions: Up, Down, Left, Right
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    while queue:
        r, c, path = queue.popleft()

        if (r, c) == goal:
            return path

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:
                if (nr, nc) not in visited:
                    visited.add((nr, nc))
                    queue.append((nr, nc, path + [(nr, nc)]))

    return None

# Test Maze
maze = [
    [0, 1, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
]

start, goal = (0, 0), (4, 4)
path = bfs_maze(maze, start, goal)

print("Shortest Path Coordinates:")
print(path)
print(f"Path Length: {len(path) - 1} steps")

Shortest Path Coordinates:
[(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3), (3, 4), (4, 4)]
Path Length: 8 steps


In [3]:
def bayes_rule(p_prior, p_likelihood, p_evidence):
    """
    P(A|B) = (P(B|A) * P(A)) / P(B)
    """
    return (p_likelihood * p_prior) / p_evidence

# Inputs
p_rain = 0.20                     # P(Rain)
p_cloudy_given_rain = 0.85        # P(Cloudy | Rain)
p_cloudy_given_not_rain = 0.30    # P(Cloudy | No Rain)

# 1. Calculate Total Probability of Evidence P(Cloudy)
p_not_rain = 1 - p_rain
p_cloudy = (p_cloudy_given_rain * p_rain) + (p_cloudy_given_not_rain * p_not_rain)

# 2. Apply Bayes' Rule P(Rain | Cloudy)
p_rain_given_cloudy = bayes_rule(
    p_prior=p_rain,
    p_likelihood=p_cloudy_given_rain,
    p_evidence=p_cloudy
)

print(f"P(Cloudy) [Total Evidence] = {p_cloudy:.4f}")
print(f"P(Rain | Cloudy) [Posterior] = {p_rain_given_cloudy:.4f} ({p_rain_given_cloudy * 100:.2f}%)")

P(Cloudy) [Total Evidence] = 0.4100
P(Rain | Cloudy) [Posterior] = 0.4146 (41.46%)


In [4]:
from collections import deque

def is_valid(m, c):
    """Checks if state (M, C) on the left bank is safe and valid."""
    if m < 0 or c < 0 or m > 3 or c > 3:
        return False
    if m > 0 and m < c:             # Left bank: Cannibals outnumber Missionaries
        return False
    if (3 - m) > 0 and (3 - m) < (3 - c): # Right bank: Cannibals outnumber Missionaries
        return False
    return True

def solve_missionaries_cannibals():
    # State format: (Missionaries_Left, Cannibals_Left, Boat_Position [1=Left, 0=Right])
    start_state = (3, 3, 1)
    goal_state = (0, 0, 0)
    
    # Possible boat moves: (M, C)
    moves = [(1, 0), (2, 0), (0, 1), (0, 2), (1, 1)]
    
    queue = deque([(start_state, [start_state])])
    visited = {start_state}

    while queue:
        (m, c, boat), path = queue.popleft()

        if (m, c, boat) == goal_state:
            return path

        for dm, dc in moves:
            if boat == 1:  # Moving Left to Right bank
                next_state = (m - dm, c - dc, 0)
            else:          # Moving Right to Left bank
                next_state = (m + dm, c + dc, 1)

            nm, nc, nb = next_state
            if is_valid(nm, nc) and next_state not in visited:
                visited.add(next_state)
                queue.append((next_state, path + [next_state]))

    return None

# Print Solution Path
solution = solve_missionaries_cannibals()
print("Missionaries & Cannibals Solution Steps:")
for step_idx, (m, c, b) in enumerate(solution):
    bank = "Left" if b == 1 else "Right"
    print(f"Step {step_idx}: Left Bank = ({m} Missionaries, {c} Cannibals) | Boat on {bank}")

Missionaries & Cannibals Solution Steps:
Step 0: Left Bank = (3 Missionaries, 3 Cannibals) | Boat on Left
Step 1: Left Bank = (3 Missionaries, 1 Cannibals) | Boat on Right
Step 2: Left Bank = (3 Missionaries, 2 Cannibals) | Boat on Left
Step 3: Left Bank = (3 Missionaries, 0 Cannibals) | Boat on Right
Step 4: Left Bank = (3 Missionaries, 1 Cannibals) | Boat on Left
Step 5: Left Bank = (1 Missionaries, 1 Cannibals) | Boat on Right
Step 6: Left Bank = (2 Missionaries, 2 Cannibals) | Boat on Left
Step 7: Left Bank = (0 Missionaries, 2 Cannibals) | Boat on Right
Step 8: Left Bank = (0 Missionaries, 3 Cannibals) | Boat on Left
Step 9: Left Bank = (0 Missionaries, 1 Cannibals) | Boat on Right
Step 10: Left Bank = (1 Missionaries, 1 Cannibals) | Boat on Left
Step 11: Left Bank = (0 Missionaries, 0 Cannibals) | Boat on Right


In [5]:
import heapq

graph = {
    'S': [('A', 1), ('B', 4)],
    'A': [('B', 2), ('C', 5), ('G', 12)],
    'B': [('C', 2)],
    'C': [('G', 3)],
    'G': []
}

heuristics = {'S': 7, 'A': 6, 'B': 4, 'C': 2, 'G': 0}

def uniform_cost_search(graph, start, goal):
    pq = [(0, start, [start])]
    visited = set()
    expansions = 0

    while pq:
        g, node, path = heapq.heappop(pq)

        if node in visited:
            continue
        visited.add(node)
        expansions += 1

        if node == goal:
            return g, path, expansions

        for neighbor, cost in graph[node]:
            if neighbor not in visited:
                heapq.heappush(pq, (g + cost, neighbor, path + [neighbor]))

def a_star_search(graph, heuristics, start, goal):
    pq = [(heuristics[start], 0, start, [start])]
    visited = set()
    expansions = 0

    while pq:
        f, g, node, path = heapq.heappop(pq)

        if node in visited:
            continue
        visited.add(node)
        expansions += 1

        if node == goal:
            return g, path, expansions

        for neighbor, cost in graph[node]:
            if neighbor not in visited:
                g_new = g + cost
                f_new = g_new + heuristics[neighbor]
                heapq.heappush(pq, (f_new, g_new, neighbor, path + [neighbor]))

# Execute & Compare
ucs_cost, ucs_path, ucs_exp = uniform_cost_search(graph, 'S', 'G')
astar_cost, astar_path, astar_exp = a_star_search(graph, heuristics, 'S', 'G')

print(f"=== Uniform Cost Search (UCS) ===")
print(f"Path: {' -> '.join(ucs_path)} | Total Cost: {ucs_cost} | Nodes Expanded: {ucs_exp}")

print(f"\n=== A* Search ===")
print(f"Path: {' -> '.join(astar_path)} | Total Cost: {astar_cost} | Nodes Expanded: {astar_exp}")

=== Uniform Cost Search (UCS) ===
Path: S -> A -> B -> C -> G | Total Cost: 8 | Nodes Expanded: 5

=== A* Search ===
Path: S -> A -> B -> C -> G | Total Cost: 8 | Nodes Expanded: 5
